# 03 · Model B: a character-level CNN

This notebook trains Model B, a character-level CNN that reads the URL itself instead of features computed from it. It measures how reproducible GPU training is, exports the model to ONNX for the app, and sets the calibrator and thresholds on validation. Model A is given 31 hand-written features; Model B has to find its own.

**Why a character CNN.** It needs no feature engineering, so it can pick up patterns nobody thought to write a feature for: character sequences typical of generated hostnames, brand names with a letter swapped, the look of a kit's path. A convolution of width 3, 5 or 7 slides over the URL and reacts to short character patterns wherever they occur. The model is small (89,633 weights, printed in section 2), so the exported file is under 1 MB and scores a URL on CPU in about a millisecond.

**Pitfalls.** A model that learns its own features will also learn dataset artefacts without being asked: a spelling habit of the collector, or hosting platforms that happened to be common in the training months. It can memorise specific hostnames. And its decisions are much harder to explain than a tree model's. That is why the project pairs it with LightGBM and reports every result separately for domains seen and not seen in training.

**Rules followed here.** Early stopping uses a split taken from inside the fitting data. The validation set is used once, to fit the calibrator and choose thresholds. No evaluation set is opened.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")     # must be set before the GPU is first used

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, roc_auc_score

from src import data as D
from src import features as F
from src import models as M

D.set_seeds(D.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

fit = D.load_processed("phreshphish_train")
val = D.load_processed("phreshphish_val")
print(f"Fitting data: {len(fit):,} URLs   Validation: {len(val):,} URLs")

torch 2.15.0+cu134 | device: cuda | NVIDIA GeForce RTX 4080 SUPER


Fitting data: 418,143 URLs   Validation: 76,009 URLs


## 1. Input encoding

`src.features.encode_chars` turns a URL into 200 integers, one per character: 0 is padding, 1 is "unknown" (anything outside printable ASCII, including decoded punycode), and 2 to 96 are the printable ASCII characters. URLs longer than 200 characters are cut; shorter ones are padded on the right. The query string is removed first (main track), exactly as for Model A.

In [2]:
example = "https://secure-login.example.net/account/verify?id=42"       # an invented URL
ids = F.encode_chars(example, strip_query=True)
form = F.normalize_url(example, strip_query=True)
display(pd.DataFrame({"character": list(form[:12]), "id": ids[:12].tolist()}).T
        .style.set_caption(f"First 12 characters of {form}"))
lengths = fit["url"].str.len()
print(f"Fitting URLs longer than 200 characters (cut): {(lengths > 200).mean():.2%}; median length {lengths.median():.0f}")

,0,1,2,3,4,5,6,7,8,9,10,11
character,h,t,t,p,s,:,/,/,s,e,c,u
id,74,86,86,82,85,28,17,17,85,71,69,87


Fitting URLs longer than 200 characters (cut): 0.24%; median length 47


## 2. Architecture

| Layer | Detail |
|---|---|
| Embedding | 97 character ids → 32 dimensions |
| Three parallel Conv1D | widths 3, 5 and 7, 128 filters each, ReLU |
| Global max-pool | one value per filter: "did this pattern occur anywhere?" |
| Concatenate | 3 × 128 = 384 values |
| Dropout | 0.3 |
| Dense | 64 units, ReLU, then dropout 0.3 |
| Output | 1 value: the raw score (log-odds) |

In [3]:
model = M.build_cnn()
print(f"Trainable weights: {sum(p.numel() for p in model.parameters()):,}")

Trainable weights: 89,633


## 3. Training

- **Optimizer:** AdamW, learning rate 0.001, weight decay 0.01, batch size 512.
- **Loss:** binary cross-entropy with the phishing class weighted by (legitimate rows / phishing rows).
- **Early stopping:** the most recent days of the fitting data (at least 5% of its rows) are set aside. After each epoch the model is scored on them, training stops when PR-AUC has not improved for 3 epochs, and the best epoch's weights are kept. A *later* slice is used because the model will be used on later data.
- **All of the remaining fitting data is used.** Training ran on a GPU; epoch times are in the table below. On a CPU the same code runs, much more slowly.

In [4]:
parts, stop_report = D.temporal_split(fit, val_share=M.CNN_TRAINING["early_stopping_share"])
train_part, stop_part = parts["train"], parts["val"]
print(f"Training rows:        {len(train_part):,}  ({stop_report['train']['date_min']} to {stop_report['train']['date_max']})")
print(f"Early-stopping split: {len(stop_part):,}  ({stop_report['val']['date_min']} to {stop_report['val']['date_max']}), "
      f"{stop_part['label'].mean():.1%} phishing")

ids_train, y_train = M.char_ids(train_part), train_part["label"].to_numpy()
ids_stop, y_stop = M.char_ids(stop_part), stop_part["label"].to_numpy()
ids_val, y_val = M.char_ids(val), val["label"].to_numpy()

Training rows:        395,207  (2024-07-02 to 2025-07-09)
Early-stopping split: 22,936  (2025-07-10 to 2025-07-28), 35.1% phishing


In [5]:
flags = M.set_torch_determinism(M.SEED)
model, history = M.train_cnn(ids_train, y_train, ids_stop, y_stop, device=device, log=lambda line: None)
pd.DataFrame(history).set_index("epoch")

,train_loss,stop_pr_auc,stop_roc_auc,seconds
epoch,,,,
1,0.171162,0.919959,0.918868,7.765716
2,0.100484,0.924748,0.920112,7.441559
3,0.081961,0.927951,0.924263,7.399869
4,0.073509,0.928568,0.923278,7.171790
5,0.067252,0.926805,0.918211,7.220708
6,0.062613,0.928811,0.923544,7.506653
7,0.059550,0.929662,0.924321,7.378346
8,0.056972,0.928990,0.921420,7.394574
9,0.054291,0.932851,0.929938,7.260533


## 4. Reproducibility on a GPU

**GPU training is not guaranteed to be bit-for-bit reproducible.** The seeds are fixed and every deterministic setting PyTorch offers is switched on (printed below). Even so, PyTorch does not promise identical results across GPU models, driver versions or library versions, and some GPU operations have no deterministic implementation. So the run-to-run difference is measured, not assumed to be zero: the training is repeated with the same seed and both models are scored on validation.

The model that is kept is **run 1**, decided before either run was scored. Run 2 exists only for this comparison.

In [6]:
display(pd.Series(flags, name="setting").to_frame().style.set_caption("Deterministic settings"))
model_2, history_2 = M.train_cnn(ids_train, y_train, ids_stop, y_stop, device=device, log=lambda line: None)

raw_1, raw_2 = M.cnn_raw(model, ids_val, device), M.cnn_raw(model_2, ids_val, device)
model, model_2 = model.cpu(), model_2.cpu()
variation = {
    "run_1": {"epochs_run": len(history), "best_epoch": int(np.argmax([h["stop_pr_auc"] for h in history])) + 1,
              "validation_roc_auc": float(roc_auc_score(y_val, raw_1)),
              "validation_pr_auc": float(average_precision_score(y_val, raw_1))},
    "run_2": {"epochs_run": len(history_2), "best_epoch": int(np.argmax([h["stop_pr_auc"] for h in history_2])) + 1,
              "validation_roc_auc": float(roc_auc_score(y_val, raw_2)),
              "validation_pr_auc": float(average_precision_score(y_val, raw_2))},
    "bit_identical": bool(np.array_equal(raw_1, raw_2)),
    "max_abs_difference_in_probability": float(np.max(np.abs(M.sigmoid(raw_1) - M.sigmoid(raw_2)))),
    "mean_abs_difference_in_probability": float(np.mean(np.abs(M.sigmoid(raw_1) - M.sigmoid(raw_2)))),
}
variation["abs_difference_roc_auc"] = abs(variation["run_1"]["validation_roc_auc"] - variation["run_2"]["validation_roc_auc"])
variation["abs_difference_pr_auc"] = abs(variation["run_1"]["validation_pr_auc"] - variation["run_2"]["validation_pr_auc"])
pd.DataFrame({"run 1 (kept)": variation["run_1"], "run 2": variation["run_2"]})

,setting
seed,42
cudnn.deterministic,True
cudnn.benchmark,False
use_deterministic_algorithms,True (warn_only)
CUBLAS_WORKSPACE_CONFIG,:4096:8


,run 1 (kept),run 2
epochs_run,15.000000,15.000000
best_epoch,12.000000,12.000000
validation_roc_auc,0.885602,0.885602
validation_pr_auc,0.852690,0.852690


In [7]:
display(Markdown(
    f"- The two runs are **{'bit-for-bit identical' if variation['bit_identical'] else 'not bit-for-bit identical'}** on validation.\n"
    f"- Validation ROC-AUC differs by **{variation['abs_difference_roc_auc']:.5f}** and PR-AUC by **{variation['abs_difference_pr_auc']:.5f}**.\n"
    f"- For a single URL, the predicted probability differs by at most {variation['max_abs_difference_in_probability']:.4f} "
    f"(mean {variation['mean_abs_difference_in_probability']:.5f}).\n\n"
    + ("On this machine the deterministic settings were enough: the two runs agree exactly. That is not guaranteed on a "
       "different GPU, driver or PyTorch version, where a rerun should reproduce the CNN results closely but perhaps not exactly. "
       if variation["bit_identical"] else
       "A rerun of this notebook on a GPU should therefore reproduce the reported CNN results to about that precision, not exactly. ")
    + "The LightGBM models and everything downstream of the saved `cnn.onnx` file are exactly reproducible."
))

- The two runs are **bit-for-bit identical** on validation.
- Validation ROC-AUC differs by **0.00000** and PR-AUC by **0.00000**.
- For a single URL, the predicted probability differs by at most 0.0000 (mean 0.00000).

On this machine the deterministic settings were enough: the two runs agree exactly. That is not guaranteed on a different GPU, driver or PyTorch version, where a rerun should reproduce the CNN results closely but perhaps not exactly. The LightGBM models and everything downstream of the saved `cnn.onnx` file are exactly reproducible.

## 5. Export to ONNX and check it against PyTorch

The app does not use PyTorch. The trained model is exported to ONNX and run with ONNX Runtime on CPU. The export is checked by scoring the same URLs with both: the two probabilities must agree to within 0.00001 for every URL. The check uses 5,000 validation URLs plus edge cases (all padding, all one character, a URL of exactly 200 characters).

In [8]:
onnx_path = M.export_onnx(model, M.MODELS_DIR / "cnn.onnx")
session = M.onnx_session(onnx_path)

rng = np.random.default_rng(M.SEED)
probe = np.vstack([
    ids_val[rng.choice(len(ids_val), 5000, replace=False)],
    np.zeros((1, 200), np.uint8), np.full((1, 200), F.UNK_ID, np.uint8), np.full((1, 200), F.VOCAB_SIZE - 1, np.uint8),
    F.encode_chars("https://example.com/" + "a" * 180, strip_query=True).astype(np.uint8)[None, :],
])
parity = M.onnx_parity(model, session, probe)
print(f"ONNX file: {onnx_path.stat().st_size / 1024:.0f} KB")
display(pd.Series(parity, name="ONNX against PyTorch").to_frame())
assert parity["max_abs_diff_probability"] < 1e-5, "ONNX export does not match PyTorch"

ONNX file: 352 KB


,ONNX against PyTorch
rows,5004.000000
max_abs_diff_probability,0.000001
max_abs_diff_raw_score,0.000006
tolerance_probability,0.000010


## 6. Calibration and thresholds

Fitted on the validation set only, from the scores of the **exported ONNX model**, since that is the model the app runs. The method is the same as for Model A (notebook 02, section 4): thresholds on the raw score, an isotonic calibrator stored as breakpoints.

A calibrated score is not a real-world probability.

In [9]:
raw_val = M.onnx_raw(session, ids_val)
training = {
    "algorithm": "character-level CNN (PyTorch), exported to ONNX",
    "architecture": M.CNN_CONFIG, "training": M.CNN_TRAINING, "device": str(torch.cuda.get_device_name(0) if device == "cuda" else "cpu"),
    "torch_version": torch.__version__,
    "fit_rows": int(len(train_part)), "early_stopping_rows": int(len(stop_part)),
    "early_stopping_split": {k: stop_report[k] for k in ("cutoff_date", "val_share")},
    "class_weight_pos_weight": float((len(y_train) - y_train.sum()) / y_train.sum()),
    "history": history, "epochs_run": len(history), "best_epoch": variation["run_1"]["best_epoch"],
    "mean_seconds_per_epoch": float(np.mean([h["seconds"] for h in history])),
    "deterministic_settings": flags, "run_to_run_variation": variation, "onnx_parity": parity,
    "validation_roc_auc": float(roc_auc_score(y_val, raw_val)),
    "validation_pr_auc": float(average_precision_score(y_val, raw_val)),
    "seed": M.SEED,
}
operating = M.build_operating_file("cnn", "main", raw_val, y_val, {"training": training})
M.save_json(operating, M.MODELS_DIR / "cnn_operating.json")
torch.save(model.state_dict(), M.MODELS_DIR / "cnn_state.pt")       # local only (gitignored); the ONNX file is the model
hashes = M.write_hashes([onnx_path, M.MODELS_DIR / "cnn_operating.json"])

## 7. Validation results

Read back from the saved files. As in notebook 02, these are validation numbers that only confirm training worked; the test results are in notebook 04.

In [10]:
saved = {name: M.load_json(M.MODELS_DIR / f"{name}_operating.json") for name in ("lgbm", "cnn")}
table = pd.DataFrame({
    label: {
        "validation ROC-AUC": f"{op['training']['validation_roc_auc']:.4f}",
        "validation PR-AUC": f"{op['training']['validation_pr_auc']:.4f}",
        "block threshold: validation false-positive rate": f"{op['thresholds']['block']['validation_fpr']:.3%}",
        "block threshold: validation detection": f"{op['thresholds']['block']['validation_tpr']:.1%}",
        "warn threshold: validation false-positive rate": f"{op['thresholds']['warn']['validation_fpr']:.2%}",
        "warn threshold: validation detection": f"{op['thresholds']['warn']['validation_tpr']:.1%}",
    } for label, (name, op) in zip(("Model A: LightGBM", "Model B: CNN"), saved.items())
})
display(table)
t = saved["cnn"]["training"]
display(Markdown(
    f"- The CNN trained for **{t['epochs_run']} epochs** (best: epoch {t['best_epoch']}) at about "
    f"**{t['mean_seconds_per_epoch']:.0f} seconds per epoch** on the {t['device']}, on {t['fit_rows']:,} URLs.\n"
    f"- Validation ROC-AUC: **{t['validation_roc_auc']:.4f}** (LightGBM: {saved['lgbm']['training']['validation_roc_auc']:.4f}).\n"
    f"- ONNX and PyTorch agree to within {t['onnx_parity']['max_abs_diff_probability']:.1e} in probability "
    f"on {t['onnx_parity']['rows']:,} URLs.\n"
    f"- Flag for a possible artefact (validation ROC-AUC above 0.999): "
    f"**{'RAISED' if t['validation_roc_auc'] > 0.999 else 'not raised'}**."
))

,Model A: LightGBM,Model B: CNN
validation ROC-AUC,0.8901,0.8856
validation PR-AUC,0.8342,0.8527
block threshold: validation false-positive rate,0.100%,0.100%
block threshold: validation detection,16.3%,44.2%
warn threshold: validation false-positive rate,65.92%,71.46%
warn threshold: validation detection,95.0%,95.0%


- The CNN trained for **15 epochs** (best: epoch 12) at about **7 seconds per epoch** on the NVIDIA GeForce RTX 4080 SUPER, on 395,207 URLs.
- Validation ROC-AUC: **0.8856** (LightGBM: 0.8901).
- ONNX and PyTorch agree to within 1.3e-06 in probability on 5,004 URLs.
- Flag for a possible artefact (validation ROC-AUC above 0.999): **not raised**.

## 8. Conclusion

Both models are now frozen: `models/lgbm.txt` and `models/cnn.onnx`, each with an `_operating.json` file holding its calibrator and thresholds, and `models/hashes.json` holding the SHA-256 of every file. On validation Model B ranks URLs about as well as Model A but catches more phishing at the block threshold; both warn thresholds flag most legitimate URLs (the table above).

Next: `04_evaluation.ipynb` scores the evaluation sample and the two external datasets, once.

## 9. Post hoc: a CNN tuning check (decision 24)

**This section was added after the test results had been seen**, on 2026-11-21, after a review of the finished project. The plan was written in `docs/design_decisions.md` (decision 24) before it was run.

Model A's settings were searched with Optuna (notebook 02). Model B's were fixed in advance and never searched. This check asks whether those fixed settings were a poor choice. The shipped architecture is trained again at three learning rates (0.0003, 0.001, 0.003) and two dropout values (0.3, 0.5), six runs, with everything else as in section 3: the same fitting data and early-stopping split, the same seed, class weights and early stopping. The shipped settings, 0.001 and 0.3, are one of the six.

Only **validation PR-AUC** is measured. No test set is scored with these models, nothing is calibrated, no threshold is set and none of the six models is saved. The rule for "among the best" was fixed before running: within 0.005 validation PR-AUC of the best run.

In [11]:
from src import posthoc as P

# Post hoc (decision 24): six settings, validation PR-AUC only. About 15 minutes on the GPU.
saved_cnn = M.load_json(M.MODELS_DIR / "cnn_operating.json")["training"]
check = P.cnn_tuning_check(ids_train, y_train, ids_stop, y_stop, ids_val, y_val, device=device,
                           shipped_validation_pr_auc=saved_cnn["validation_pr_auc"])
M.save_json(check, P.CNN_CHECK_FILE)
(pd.DataFrame(check["runs"])
   .assign(shipped=lambda d: d["shipped_settings"].map({True: "shipped", False: ""}))
   .set_index(["learning_rate", "dropout"])[["validation_pr_auc", "rank", "epochs_run", "best_epoch", "shipped"]]
   .style.format({"validation_pr_auc": "{:.4f}"})
   .set_caption("Validation PR-AUC by setting (post hoc; no test set scored)"))

learning rate 0.0003 dropout 0.3  epochs 15 (best 12)  validation PR-AUC 0.8450


learning rate 0.0003 dropout 0.5  epochs 23 (best 20)  validation PR-AUC 0.8442


learning rate 0.001  dropout 0.3  epochs 15 (best 12)  validation PR-AUC 0.8527


learning rate 0.001  dropout 0.5  epochs 12 (best  9)  validation PR-AUC 0.8465


learning rate 0.003  dropout 0.3  epochs 14 (best 11)  validation PR-AUC 0.8483


learning rate 0.003  dropout 0.5  epochs 14 (best 11)  validation PR-AUC 0.8489


In [12]:
s, b = check["shipped"], check["best"]
display(Markdown(
    f"- **Shipped settings** (learning rate {s['learning_rate']:g}, dropout {s['dropout']}): validation PR-AUC "
    f"**{s['validation_pr_auc_retrained']:.4f}**, rank {s['rank']} of {len(check['runs'])}. "
    f"The saved model's validation PR-AUC, from the exported ONNX file, is {s['validation_pr_auc_saved']:.4f}.\n"
    f"- **Best of the six** (learning rate {b['learning_rate']:g}, dropout {b['dropout']}): "
    f"**{b['validation_pr_auc']:.4f}**, {check['gap_best_minus_shipped']:.4f} above the shipped settings.\n"
    f"- **Were the shipped settings among the best** (within 0.005 of the best)? "
    f"**{'Yes' if check['shipped_among_best'] else 'No'}.**\n"
    f"- Saved to `reports/cnn_tuning_check.json`."
))

- **Shipped settings** (learning rate 0.001, dropout 0.3): validation PR-AUC **0.8527**, rank 1 of 6. The saved model's validation PR-AUC, from the exported ONNX file, is 0.8527.
- **Best of the six** (learning rate 0.001, dropout 0.3): **0.8527**, 0.0000 above the shipped settings.
- **Were the shipped settings among the best** (within 0.005 of the best)? **Yes.**
- Saved to `reports/cnn_tuning_check.json`.

**The deployed CNN is kept, whatever this table shows.** Choosing a different model now, after the test results have been seen, would let those results influence the model and bias every test number reported for it. The check answers one question for the record, whether the fixed settings were a poor choice; it does not select a model.